# L2C - Pipeline E2E Plan vs DA

Même flux que `src/pipeline.py` : extraction du plan, extraction des DA (object finder), audit, rapport PDF. Installer avec `uv sync --dev` et choisir le Python de `.venv`.

## 1. Configuration

Indiquer le dossier du projet. Le plan est le PDF dont le nom contient `PLAN`; tous les autres PDF du dossier (sous-dossiers inclus) sont traités comme des DA. Pour forcer le plan, renseigner `PLAN`. `PAGES = None` traite tout le plan.

In [2]:
from pathlib import Path
import json

ROOT = Path.cwd()
FOLDER = Path("data/CLP")  # ex. l2c-participants/CLP
PLAN = None  # ex. FOLDER / "L2C_PLAN_STR_CLP.pdf"; None = détecté
PAGES = None  # ex. [4, 14, 18, 26]
PROJET = FOLDER.name or "CLP"
OUT = ROOT / "out/notebook" / PROJET

from llm_parser.object_finder import find_all_objects_in_pdf
from src.extract import extract_pdf
from src.match import match_elements
from src.report import audit_report

pdfs = sorted(FOLDER.rglob("*.pdf"))
if PLAN is None:
    plans = [p for p in pdfs if "plan" in p.stem.lower()]
    if len(plans) != 1:
        raise ValueError(f"Plan introuvable ou ambigu dans {FOLDER}: {[p.name for p in plans]}. Renseigner PLAN.")
    PLAN = plans[0]
DA_PATHS = [p for p in pdfs if p != Path(PLAN)]
if not DA_PATHS:
    raise ValueError(f"Aucun PDF DA trouvé dans {FOLDER}")

OUT.mkdir(parents=True, exist_ok=True)
print("Plan:", PLAN)
print("DA:", *[p.name for p in DA_PATHS], sep="\n  ")

Plan: data/CLP/L2C_PLAN_STR_CLP.pdf
DA:
  CLP_COLONNES Partie 1.pdf
  CLP_COLONNES Partie 2.pdf
  CLP_COLONNES Partie 3.pdf
  CLP_DALLE NIV 2.pdf
  CLP_DALLE NIV 3.pdf
  CLP_DALLE NIV 4.pdf
  CLP_DALLE NIV 5.pdf
  CLP_DALLE NIV RDC.pdf
  CLP_DALLE TRÉFOND.pdf
  CLP_RADIERS.pdf
  CLP_SEMELLES FND.pdf
  CLP_POUTRES.pdf


## 2. Extraction du plan

Lit le plan (PyMuPDF) et parse les annotations d’armature.

In [3]:
plan_records = extract_pdf(Path(PLAN), pages=PAGES)
plan_json = OUT / "plan.json"
plan_json.write_text(json.dumps(plan_records, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Plan: {len(plan_records)} éléments -> {plan_json}")

Plan: 1549 éléments -> /Users/adrientrahan/Desktop/codeml/out/notebook/CLP/plan.json


## 3. Extraction des DA

Object finder sur chaque DA.

In [4]:
da_records: list[dict] = []
for da_pdf in DA_PATHS:
    recs = find_all_objects_in_pdf(da_pdf)
    da_records.extend(recs)
    print(f"DA {da_pdf.name}: {len(recs)} éléments")
da_json = OUT / "da.json"
da_json.write_text(json.dumps(da_records, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"DA total: {len(da_records)} éléments -> {da_json}")

0it [00:00, ?it/s]
100%|██████████| 17/17 [00:47<00:00,  2.81s/it]


FileNotFoundError: [Errno 2] No such file or directory: 'data/CLP/DA/Colonnes/CLP_COLONNES Partie 1.pdf'

## 4. Audit

Compare les éléments du plan et des DA.

In [ ]:
discrepancies = match_elements(plan_records, da_records)
disc_json = OUT / "discrepancies.json"
disc_json.write_text(
    json.dumps([d.to_dict() for d in discrepancies], indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Audit: {len(discrepancies)} éléments -> {disc_json}")

## 5. Rapport PDF

In [ ]:
audit_pdf = OUT / "audit.pdf"
audit_report(discrepancies, audit_pdf, projet=PROJET)
print(f"Rapport: {audit_pdf}")
{"plan": len(plan_records), "da": len(da_records), "discrepancies": len(discrepancies)}